# Detekce mutací <i>Plasmodia falciparum</i> pomocí bioinformatických nástrojů v příkazové řádce
## Obsah hodiny


## Motivace: Proč nás zajímají mutace plasmodia?
TODO: nějaký obrázek, malárie, rezistence, geny


## Co je to příkazová řádka?

<b>GUI = graphic user interface = grafické rozhraní</b>

Ovládání počítače pomocí interaktivních grafických prvků jako například okna, menu, klikací ikony

<b>CLI = command line interface = příkazová řádka (terminál) </b>

Ovládání počítače pomocí textových příkazů 

<details>
  <summary>Q: Proč používat příkazovou řádku pro řešení bioinformatických úkolů?</summary>
   
+ flexibilita v kombinování existujících nástrojů a možnost zapojení vlastních skriptů
+ zpracování velkého množství souborů najednou

</details>

<br/><br/>
<b>Jak otevřít příkazovou řádku v Jupyter Lab</b>

V horním menu klikněte na <b>File</b> > <b>New</b> > <b>Terminal</b>

Vyzkoušejte zadat příkaz do příkazové řádky (potvrdít stisknutím <b>Enter<b>):

`ls` - vypsat soubory v aktuálním adresáři

`pwd` - vypsat cestu do aktuálního adresáře

`date` - vypsat aktuální systémový čas

Nebo můžete spustit příkaz přímo v buňkách s kódem v tomto notebooku:  

In [1]:
!ls # označte tuto buňku šipkami na klávesnici (nebo levým tlačítkem myši) a stiskněte Shift+Enter na klávesnici (nebo ikonu "Run this cell and advance" na horní liště)
# můžete taky obsah buňky přepsat - vyzkoušejte, jak se výstup příkazu změní, když u něj bude doplňkový parametr - jako například ls -la

ERR042228_F.fq				      detekce_variant_v_plasmodiu.ipynb
ERR042228_R.fq				      galaxy_inputs
GCF_000002765.6.fa			      ipython_galaxy_notebook.ipynb
GCF_000002765.6_GCA_000002765.ncbiRefSeq.gtf  outputs
ansible_kernel.log


<b>Poznámka:</b> `!` na začátku buňky říká Jupyter Labu "spusť kód jako příkaz v terminálu". Když do buňky napíšeš kód bez vykřičníku: `print("ahoj")`, poběží to jako python kód. Příkazová řádka a python používají jinou "gramatiku" takže je potřeba odlišit, jak se má příkaz spustit. 

In [2]:
print("ahoj pythone!")
!echo "ahoj příkazová řádko!"

ahoj pythone!
ahoj příkazová řádko!


## Stažení vstupních souborů
potřebujeme: 
- sekvence DNA plasmodia z pacientů - krátká čtení
- referenční sekvenci plasmodia - 
- pozici genu <i>dhfr</i> v referenční sekvenci (tzv. genové anotace)

Všechny potřebné soubory pro toto cvičení (původně vytvořeny pro [tuto Galaxy lekci](https://training.galaxyproject.org/training-material/topics/introduction/tutorials/galaxy-intro-ngs-data-managment/tutorial.html#annotating-variants{target=_blank})) jsou dostupné v internetovém archivu, ze kterého si je můžeme stáhnout pomocí příkazu `wget`. 

In [3]:
!wget https://zenodo.org/records/15354240/files/ERR042228_F.fq.gz # soubor obsahující DNA čtení (DNA reads)  - formát fastq
!wget https://zenodo.org/records/15354240/files/ERR042228_R.fq.gz # soubor obsahující DNA čtení (DNA reads) - formát fastq
!wget https://zenodo.org/records/15354240/files/GCF_000002765.6.fa.gz # referenční sekvence plasmodium falciparum - formát fasta
!wget https://zenodo.org/records/15354240/files/GCF_000002765.6_GCA_000002765.ncbiRefSeq.gtf.gz # genové anotace - formát gtf

Will not apply HSTS. The HSTS database must be a regular and non-world-writable file.
ERROR: could not open HSTS store at '/home/jovyan//.wget-hsts'. HSTS will be disabled.
--2026-02-05 17:04:29--  https://zenodo.org/records/15354240/files/ERR042228_F.fq.gz
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.153.219, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2973476 (2.8M) [application/octet-stream]
Saving to: ‘ERR042228_F.fq.gz’

ERR042228_F.fq.gz   100%[===================>]   2.83M  --.-KB/s    in 0.1s    

2026-02-05 17:04:29 (25.1 MB/s) - ‘ERR042228_F.fq.gz’ saved [2973476/2973476]

Will not apply HSTS. The HSTS database must be a regular and non-world-writable file.
ERROR: could not open HSTS store at '/home/jovyan//.wget-hsts'. HSTS will be disabled.
--2026-02-05 17:04:29--  https://zenodo.org/records/15354240/files/ERR042228_R.fq.gz
Resolving zenodo.org (zenodo.org)

Stažené soubory mají příponu `.gz`, jsou tedy komprimované za účelem zmenšení jejich velikosti. Abychom s nimi mohli snadno pracovat, je nutné je nejdřív rozbalit pomocí `gunzip`.

<b>Poznámka:</b> `*` je takzvaný "žolíkový znak" (wildcard), která zastupuje libovolné znaky. Takže příkaz níže rozbalí všechny soubory uvnitř aktuální složky, které končí `.gz` a není potřeba vypisovat názvy jednotlivých souborů.

In [ ]:
!gunzip *.gz

gzip: ERR042228_F.fq already exists; do you wish to overwrite (y or n)? 

## Mapování sekvencí na referenční genom

Nejdřív si musíme nainstalovat nástroj pro mapování. Ten se nazývá BWA (Burrows-Wheeler Alignment) a funguje na základě hledání krátkých sekvencí v DNA readech, které se přesně shodují referencí. Aby bylo toto hledání rychlé, nástroj nejříve DNA sekvenci algoritmicky přetransformuje do podoby, ve které se tyto podobnosti hledají snáz - takzvaného indexu.
<b>Poznámka:</b> 

In [1]:
!mamba install bioconda::bwa --yes

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
bioconda/linux-64                                           Using cache
bioconda/noarch                                             Using cache

Pinned packages:

  - python=3.12

Pinned packages:

  - python=3.12


Transaction

  Prefix: /opt/conda

  All requested packages already installed


Transaction starting
[+] 0.0s

Transaction finished



## Acknowledgement
tvůrci původního GT